
# ساختار پروژه

```
.
├── data/
│   └── Wholesale_Customers_Data.csv  (یا نام‌های مشابه)
├── outputs/
│   ├── figures/
│   ├── tables/
│   ├── data_splits/
│   └── models/
└── 00_Wholesale_Linear_Models_Full.ipynb
```



# دفترچه جامع مدل‌های رگرسیون خطی برای دیتاست Wholesale

**هدف:** اجرای کامل EDA، پیش‌پردازش، آموزش چند مدل رگرسیونی، ارزیابی، و مقایسه نهایی در یک نوت‌بوک.

## فهرست مطالب
1. بارگذاری داده
2. آماده‌سازی و انتخاب متغیر هدف
3. اکتشاف داده (EDA)
4. پیش‌پردازش مشترک
5. ساخت تقسیم داده و ذخیره آن
6. آموزش و ارزیابی مدل‌ها (Linear, Ridge, Lasso, ElasticNet, SGD)
7. مقایسه نهایی مدل‌ها
8. جمع‌بندی نهایی


In [ ]:

import os
import glob
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, KFold, cross_val_score, GridSearchCV, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, PowerTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet, SGDRegressor
import joblib

np.random.seed(42)

plt.rcParams["font.family"] = "DejaVu Sans"
plt.rcParams["axes.unicode_minus"] = False


In [ ]:

# تابع بارگذاری داده با پشتیبانی از چند نام فایل

def load_wholesale_data():
    candidates = [
        "Wholesale_customers_data.csv",
        "Wholesale customers data.csv",
        "wholesale_customers_data.xlsx",
        "Wholesale_Customers_Data.csv",
        "Wholesale customers data.xlsx",
    ]
    paths = []
    for name in candidates:
        paths.extend(glob.glob(os.path.join("data", name)))
    if not paths:
        raise FileNotFoundError(
            "فایل دیتاست پیدا نشد. لطفاً یکی از نام‌های مجاز را در پوشه data قرار دهید."
        )
    path = paths[0]
    print(f"فایل انتخاب‌شده: {path}")
    if path.lower().endswith(".xlsx"):
        df = pd.read_excel(path)
    else:
        df = pd.read_csv(path)
    return df


# انتخاب هدف بر اساس قوانین مسئله

def select_target(df):
    if "Grocery" in df.columns:
        target = "Grocery"
        reason = "ستون Grocery موجود است و یک متغیر هزینه‌ای مناسب برای رگرسیون محسوب می‌شود."
    elif "Detergents_Paper" in df.columns:
        target = "Detergents_Paper"
        reason = "ستون Grocery وجود ندارد؛ بنابراین از Detergents_Paper به عنوان هدف استفاده می‌کنیم."
    else:
        numeric_cols = df.select_dtypes(include=[np.number]).columns
        df["TotalSpend"] = df[numeric_cols].sum(axis=1)
        target = "TotalSpend"
        reason = "ستون‌های اصلی هدف موجود نبود؛ مجموع هزینه‌های عددی را به عنوان TotalSpend ساختیم."
    return target, reason, df


# ساخت پیش‌پردازش مشترک با بررسی skewness

def build_preprocessor(X):
    numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
    categorical_features = X.select_dtypes(exclude=[np.number]).columns.tolist()

    skewness = X[numeric_features].skew().sort_values(ascending=False) if numeric_features else pd.Series(dtype=float)
    skewed_features = skewness[skewness.abs() > 1.5].index.tolist()

    use_power = len(skewed_features) > 0

    numeric_steps = [
        ("imputer", SimpleImputer(strategy="median")),
    ]
    if use_power:
        numeric_steps.append(("power", PowerTransformer(method="yeo-johnson")))
    numeric_steps.append(("scaler", StandardScaler()))

    numeric_transformer = Pipeline(steps=numeric_steps)

    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_features),
            ("cat", categorical_transformer, categorical_features)
        ],
        remainder="drop"
    )

    info = {
        "numeric_features": numeric_features,
        "categorical_features": categorical_features,
        "skewness": skewness,
        "skewed_features": skewed_features,
        "use_power_transformer": use_power,
    }
    return preprocessor, info


# محاسبه متریک‌ها

def regression_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred, squared=False)
    r2 = r2_score(y_true, y_pred)
    return mae, rmse, r2


# توابع نمودارسازی

def plot_predicted_vs_actual(y_true, y_pred, title, filename):
    plt.figure(figsize=(6, 5))
    plt.scatter(y_true, y_pred, alpha=0.7)
    plt.plot([y_true.min(), y_true.max()], [y_true.min(), y_true.max()], "r--")
    plt.title(title)
    plt.xlabel("مقدار واقعی")
    plt.ylabel("مقدار پیش‌بینی‌شده")
    plt.tight_layout()
    plt.savefig(filename)
    plt.show()
    print("توضیح: هرچه نقاط به خط قرمز نزدیک‌تر باشند، عملکرد بهتر است.")


def plot_residuals(y_true, y_pred, title, filename):
    residuals = y_true - y_pred
    plt.figure(figsize=(6, 5))
    plt.scatter(y_pred, residuals, alpha=0.7)
    plt.axhline(0, color="r", linestyle="--")
    plt.title(title)
    plt.xlabel("مقدار پیش‌بینی‌شده")
    plt.ylabel("Residual")
    plt.tight_layout()
    plt.savefig(filename)
    plt.show()
    print("توضیح: توزیع یکنواخت حول صفر نشان‌دهنده مناسب بودن مدل است.")


def plot_residual_hist(residuals, title, filename):
    plt.figure(figsize=(6, 5))
    plt.hist(residuals, bins=30, alpha=0.7)
    plt.title(title)
    plt.xlabel("Residual")
    plt.ylabel("فراوانی")
    plt.tight_layout()
    plt.savefig(filename)
    plt.show()
    print("توضیح: شکل زنگوله‌ای و مرکز نزدیک صفر مطلوب است.")


def plot_top_coefficients(model, feature_names, title, filename, top_n=15):
    coefs = pd.Series(model.coef_.ravel(), index=feature_names)
    top = coefs.abs().sort_values(ascending=False).head(top_n).index
    plt.figure(figsize=(8, 5))
    coefs.loc[top].sort_values().plot(kind="barh")
    plt.title(title)
    plt.xlabel("مقدار ضریب")
    plt.tight_layout()
    plt.savefig(filename)
    plt.show()
    print("توضیح: ویژگی‌هایی با قدر مطلق بالاتر اثر بیشتری دارند.")


# ذخیره جدول

def save_table(df, filename):
    os.makedirs("outputs/tables", exist_ok=True)
    df.to_csv(filename, index=False)
    display(df)


# ثبت نتایج در فایل نتایج

def save_results(name, results, path="outputs/tables/results.json"):
    os.makedirs("outputs/tables", exist_ok=True)
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
    else:
        data = {}
    data[name] = results
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)


## 1) بارگذاری داده


In [ ]:
df = load_wholesale_data()
print("ستون‌ها:", df.columns.tolist())
print("انواع داده‌ها:")
print(df.dtypes)


## 2) آماده‌سازی و انتخاب متغیر هدف


In [ ]:
target, reason, df = select_target(df)
print("متغیر هدف انتخاب‌شده:", target)
print("توجیه انتخاب:", reason)

X = df.drop(columns=[target])
y = df[target]


## 3) اکتشاف داده (EDA)


In [ ]:
display(df.head())


In [ ]:
display(df.describe(include='all'))


In [ ]:
print(df.info())


In [ ]:
# Missing values
missing = df.isnull().sum()
missing_df = pd.DataFrame({"ستون": missing.index, "تعداد Missing": missing.values})
summary_table = pd.DataFrame({
    "تعداد سطر": [df.shape[0]],
    "تعداد ستون": [df.shape[1]],
    "تعداد Missing": [missing.sum()],
    "تعداد عددی": [df.select_dtypes(include=[np.number]).shape[1]],
    "تعداد کتگوریک": [df.select_dtypes(exclude=[np.number]).shape[1]],
})

save_table(summary_table, "outputs/tables/eda_summary.csv")
display(missing_df)


In [ ]:
# Boxplot برای شناسایی Outlier
numeric_cols = df.select_dtypes(include=[np.number]).columns
plt.figure(figsize=(10, 6))
df[numeric_cols].plot(kind='box', subplots=True, layout=(1, len(numeric_cols)), figsize=(15, 5))
plt.suptitle("نمودار جعبه‌ای برای ستون‌های عددی")
plt.tight_layout()
plt.savefig("outputs/figures/eda_boxplot.png")
plt.show()
print("توضیح: نقاط دور از جعبه‌ها می‌توانند به عنوان outlier در نظر گرفته شوند.")


In [ ]:
# Correlation heatmap
corr = df.select_dtypes(include=[np.number]).corr()
plt.figure(figsize=(8, 6))
plt.imshow(corr, cmap='coolwarm', interpolation='none')
plt.colorbar()
plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
plt.yticks(range(len(corr.columns)), corr.columns)
plt.title("نقشه حرارتی همبستگی")
plt.tight_layout()
plt.savefig("outputs/figures/eda_corr_heatmap.png")
plt.show()
print("توضیح: شدت رنگ نشان‌دهنده میزان همبستگی است.")


In [ ]:
# Pairwise scatter برای چند ستون مهم
cols = df.select_dtypes(include=[np.number]).columns[:4]
if len(cols) >= 2:
    pd.plotting.scatter_matrix(df[cols], figsize=(8, 6))
    plt.suptitle("نمودارهای پراکندگی زوجی")
    plt.tight_layout()
    plt.savefig("outputs/figures/eda_scatter_matrix.png")
    plt.show()
    print("توضیح: روابط خطی احتمالی بین ویژگی‌ها قابل مشاهده است.")


## 4) پیش‌پردازش مشترک


In [ ]:
preprocessor, prep_info = build_preprocessor(X)
print("ویژگی‌های عددی:", prep_info["numeric_features"])
print("ویژگی‌های کتگوریک:", prep_info["categorical_features"])
print("ویژگی‌های با چولگی بالا:", prep_info["skewed_features"])
print("استفاده از PowerTransformer:", prep_info["use_power_transformer"])


## 5) تقسیم داده و ذخیره‌سازی


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

os.makedirs("outputs/data_splits", exist_ok=True)
joblib.dump({
    "X_train": X_train,
    "X_test": X_test,
    "y_train": y_train,
    "y_test": y_test,
    "target": target,
    "prep_info": prep_info,
}, "outputs/data_splits/data_splits.pkl")

print("تقسیم داده ذخیره شد.")


## 6) آموزش و ارزیابی مدل‌ها


### 6-1) Linear Regression


In [ ]:
model = LinearRegression()
pipe = Pipeline(steps=[("preprocessor", preprocessor), ("model", model)])
pipe.fit(X_train, y_train)

train_pred = pipe.predict(X_train)
test_pred = pipe.predict(X_test)

train_mae, train_rmse, train_r2 = regression_metrics(y_train, train_pred)
test_mae, test_rmse, test_r2 = regression_metrics(y_test, test_pred)

cv = KFold(n_splits=5, shuffle=True, random_state=42)
cv_rmse = cross_val_score(pipe, X_train, y_train, scoring="neg_root_mean_squared_error", cv=cv)
cv_r2 = cross_val_score(pipe, X_train, y_train, scoring="r2", cv=cv)

results_df = pd.DataFrame({
    "Metric": ["MAE", "RMSE", "R2"],
    "Train": [train_mae, train_rmse, train_r2],
    "Test": [test_mae, test_rmse, test_r2],
})

cv_summary = pd.DataFrame({
    "Metric": ["RMSE_CV", "R2_CV"],
    "Mean": [(-cv_rmse).mean(), cv_r2.mean()],
    "Std": [(-cv_rmse).std(), cv_r2.std()],
})

save_table(results_df, "outputs/tables/linear_results.csv")
save_table(cv_summary, "outputs/tables/linear_cv_summary.csv")

save_results("LinearRegression", {
    "train_mae": train_mae,
    "train_rmse": train_rmse,
    "train_r2": train_r2,
    "test_mae": test_mae,
    "test_rmse": test_rmse,
    "test_r2": test_r2,
    "cv_rmse_mean": (-cv_rmse).mean(),
    "cv_rmse_std": (-cv_rmse).std(),
    "cv_r2_mean": cv_r2.mean(),
    "cv_r2_std": cv_r2.std(),
    "best_params": None,
})

residuals = y_test - test_pred
plot_predicted_vs_actual(y_test, test_pred, "واقعی در برابر پیش‌بینی (Linear)", "outputs/figures/linear_pred_vs_actual.png")
plot_residuals(y_test, test_pred, "Residuals (Linear)", "outputs/figures/linear_residuals.png")
plot_residual_hist(residuals, "Histogram Residuals (Linear)", "outputs/figures/linear_residual_hist.png")

feature_names = pipe.named_steps["preprocessor"].get_feature_names_out()
plot_top_coefficients(pipe.named_steps["model"], feature_names, "Top 15 ضرایب (Linear)", "outputs/figures/linear_top_coefs.png")


### 6-2) Ridge


In [ ]:
model = Ridge(random_state=42)
pipe = Pipeline(steps=[("preprocessor", preprocessor), ("model", model)])

param_grid = {"model__alpha": np.logspace(-4, 4, 50)}
cv = KFold(n_splits=5, shuffle=True, random_state=42)

grid = GridSearchCV(pipe, param_grid=param_grid, scoring="neg_root_mean_squared_error", cv=cv)
grid.fit(X_train, y_train)

best_model = grid.best_estimator_
print("بهترین alpha:", grid.best_params_)

train_pred = best_model.predict(X_train)
test_pred = best_model.predict(X_test)

train_mae, train_rmse, train_r2 = regression_metrics(y_train, train_pred)
test_mae, test_rmse, test_r2 = regression_metrics(y_test, test_pred)

cv_rmse = -grid.best_score_
cv_r2 = cross_val_score(best_model, X_train, y_train, scoring="r2", cv=cv)

results_df = pd.DataFrame({
    "Metric": ["MAE", "RMSE", "R2"],
    "Train": [train_mae, train_rmse, train_r2],
    "Test": [test_mae, test_rmse, test_r2],
})

cv_summary = pd.DataFrame({
    "Metric": ["RMSE_CV", "R2_CV"],
    "Mean": [cv_rmse, cv_r2.mean()],
    "Std": [0.0, cv_r2.std()],
})

save_table(results_df, "outputs/tables/ridge_results.csv")
save_table(cv_summary, "outputs/tables/ridge_cv_summary.csv")

save_results("Ridge", {
    "train_mae": train_mae,
    "train_rmse": train_rmse,
    "train_r2": train_r2,
    "test_mae": test_mae,
    "test_rmse": test_rmse,
    "test_r2": test_r2,
    "cv_rmse_mean": cv_rmse,
    "cv_rmse_std": 0.0,
    "cv_r2_mean": cv_r2.mean(),
    "cv_r2_std": cv_r2.std(),
    "best_params": grid.best_params_,
})

residuals = y_test - test_pred
plot_predicted_vs_actual(y_test, test_pred, "واقعی در برابر پیش‌بینی (Ridge)", "outputs/figures/ridge_pred_vs_actual.png")
plot_residuals(y_test, test_pred, "Residuals (Ridge)", "outputs/figures/ridge_residuals.png")
plot_residual_hist(residuals, "Histogram Residuals (Ridge)", "outputs/figures/ridge_residual_hist.png")

feature_names = best_model.named_steps["preprocessor"].get_feature_names_out()
plot_top_coefficients(best_model.named_steps["model"], feature_names, "Top 15 ضرایب (Ridge)", "outputs/figures/ridge_top_coefs.png")

alphas = np.logspace(-4, 4, 50)
coefs = []
for a in alphas:
    ridge = Ridge(alpha=a, random_state=42)
    pipe = Pipeline(steps=[("preprocessor", preprocessor), ("model", ridge)])
    pipe.fit(X_train, y_train)
    coefs.append(pipe.named_steps["model"].coef_.ravel())

plt.figure(figsize=(7, 5))
plt.plot(alphas, coefs)
plt.xscale('log')
plt.title("مسیر ضرایب Ridge")
plt.xlabel("alpha")
plt.ylabel("مقدار ضریب")
plt.tight_layout()
plt.savefig("outputs/figures/ridge_path.png")
plt.show()
print("توضیح: با افزایش alpha ضرایب کوچک‌تر می‌شوند.")


In [ ]:

# ذخیره مدل
os.makedirs("outputs/models", exist_ok=True)
joblib.dump(best_model, "outputs/models/ridge.pkl")
print("مدل Ridge ذخیره شد.")


### 6-3) Lasso


In [ ]:
model = Lasso(random_state=42, max_iter=5000)
pipe = Pipeline(steps=[("preprocessor", preprocessor), ("model", model)])

param_grid = {"model__alpha": np.logspace(-4, 4, 50)}
cv = KFold(n_splits=5, shuffle=True, random_state=42)

grid = GridSearchCV(pipe, param_grid=param_grid, scoring="neg_root_mean_squared_error", cv=cv)
grid.fit(X_train, y_train)

best_model = grid.best_estimator_
print("بهترین alpha:", grid.best_params_)

train_pred = best_model.predict(X_train)
test_pred = best_model.predict(X_test)

train_mae, train_rmse, train_r2 = regression_metrics(y_train, train_pred)
test_mae, test_rmse, test_r2 = regression_metrics(y_test, test_pred)

cv_rmse = -grid.best_score_
cv_r2 = cross_val_score(best_model, X_train, y_train, scoring="r2", cv=cv)

results_df = pd.DataFrame({
    "Metric": ["MAE", "RMSE", "R2"],
    "Train": [train_mae, train_rmse, train_r2],
    "Test": [test_mae, test_rmse, test_r2],
})

cv_summary = pd.DataFrame({
    "Metric": ["RMSE_CV", "R2_CV"],
    "Mean": [cv_rmse, cv_r2.mean()],
    "Std": [0.0, cv_r2.std()],
})

save_table(results_df, "outputs/tables/lasso_results.csv")
save_table(cv_summary, "outputs/tables/lasso_cv_summary.csv")

coefs = best_model.named_steps["model"].coef_.ravel()
zeros = np.sum(coefs == 0)
nonzeros = np.sum(coefs != 0)
print("تعداد ضرایب صفر:", zeros)
print("تعداد ضرایب غیرصفر:", nonzeros)

save_results("Lasso", {
    "train_mae": train_mae,
    "train_rmse": train_rmse,
    "train_r2": train_r2,
    "test_mae": test_mae,
    "test_rmse": test_rmse,
    "test_r2": test_r2,
    "cv_rmse_mean": cv_rmse,
    "cv_rmse_std": 0.0,
    "cv_r2_mean": cv_r2.mean(),
    "cv_r2_std": cv_r2.std(),
    "best_params": grid.best_params_,
    "zeros": int(zeros),
    "nonzeros": int(nonzeros),
})

residuals = y_test - test_pred
plot_predicted_vs_actual(y_test, test_pred, "واقعی در برابر پیش‌بینی (Lasso)", "outputs/figures/lasso_pred_vs_actual.png")
plot_residuals(y_test, test_pred, "Residuals (Lasso)", "outputs/figures/lasso_residuals.png")
plot_residual_hist(residuals, "Histogram Residuals (Lasso)", "outputs/figures/lasso_residual_hist.png")

feature_names = best_model.named_steps["preprocessor"].get_feature_names_out()
plot_top_coefficients(best_model.named_steps["model"], feature_names, "Top 15 ضرایب (Lasso)", "outputs/figures/lasso_top_coefs.png")

alphas = np.logspace(-4, 4, 50)
coefs = []
for a in alphas:
    lasso = Lasso(alpha=a, random_state=42, max_iter=5000)
    pipe = Pipeline(steps=[("preprocessor", preprocessor), ("model", lasso)])
    pipe.fit(X_train, y_train)
    coefs.append(pipe.named_steps["model"].coef_.ravel())

plt.figure(figsize=(7, 5))
plt.plot(alphas, coefs)
plt.xscale('log')
plt.title("مسیر ضرایب Lasso")
plt.xlabel("alpha")
plt.ylabel("مقدار ضریب")
plt.tight_layout()
plt.savefig("outputs/figures/lasso_path.png")
plt.show()
print("توضیح: Lasso با افزایش alpha ضرایب را به صفر نزدیک می‌کند.")


In [ ]:

# ذخیره مدل
os.makedirs("outputs/models", exist_ok=True)
joblib.dump(best_model, "outputs/models/lasso.pkl")
print("مدل Lasso ذخیره شد.")


### 6-4) ElasticNet


In [ ]:
model = ElasticNet(random_state=42, max_iter=5000)
pipe = Pipeline(steps=[("preprocessor", preprocessor), ("model", model)])

param_grid = {
    "model__alpha": np.logspace(-4, 4, 30),
    "model__l1_ratio": [0.1, 0.3, 0.5, 0.7, 0.9, 0.95, 0.99]
}
cv = KFold(n_splits=5, shuffle=True, random_state=42)

grid = GridSearchCV(pipe, param_grid=param_grid, scoring="neg_root_mean_squared_error", cv=cv)
grid.fit(X_train, y_train)

best_model = grid.best_estimator_
print("بهترین پارامترها:", grid.best_params_)

train_pred = best_model.predict(X_train)
test_pred = best_model.predict(X_test)

train_mae, train_rmse, train_r2 = regression_metrics(y_train, train_pred)
test_mae, test_rmse, test_r2 = regression_metrics(y_test, test_pred)

cv_rmse = -grid.best_score_
cv_r2 = cross_val_score(best_model, X_train, y_train, scoring="r2", cv=cv)

results_df = pd.DataFrame({
    "Metric": ["MAE", "RMSE", "R2"],
    "Train": [train_mae, train_rmse, train_r2],
    "Test": [test_mae, test_rmse, test_r2],
})

cv_summary = pd.DataFrame({
    "Metric": ["RMSE_CV", "R2_CV"],
    "Mean": [cv_rmse, cv_r2.mean()],
    "Std": [0.0, cv_r2.std()],
})

save_table(results_df, "outputs/tables/elasticnet_results.csv")
save_table(cv_summary, "outputs/tables/elasticnet_cv_summary.csv")

coefs = best_model.named_steps["model"].coef_.ravel()
zeros = np.sum(coefs == 0)
nonzeros = np.sum(coefs != 0)
print("تعداد ضرایب صفر:", zeros)
print("تعداد ضرایب غیرصفر:", nonzeros)

save_results("ElasticNet", {
    "train_mae": train_mae,
    "train_rmse": train_rmse,
    "train_r2": train_r2,
    "test_mae": test_mae,
    "test_rmse": test_rmse,
    "test_r2": test_r2,
    "cv_rmse_mean": cv_rmse,
    "cv_rmse_std": 0.0,
    "cv_r2_mean": cv_r2.mean(),
    "cv_r2_std": cv_r2.std(),
    "best_params": grid.best_params_,
    "zeros": int(zeros),
    "nonzeros": int(nonzeros),
})

residuals = y_test - test_pred
plot_predicted_vs_actual(y_test, test_pred, "واقعی در برابر پیش‌بینی (ElasticNet)", "outputs/figures/elasticnet_pred_vs_actual.png")
plot_residuals(y_test, test_pred, "Residuals (ElasticNet)", "outputs/figures/elasticnet_residuals.png")
plot_residual_hist(residuals, "Histogram Residuals (ElasticNet)", "outputs/figures/elasticnet_residual_hist.png")

feature_names = best_model.named_steps["preprocessor"].get_feature_names_out()
plot_top_coefficients(best_model.named_steps["model"], feature_names, "Top 15 ضرایب (ElasticNet)", "outputs/figures/elasticnet_top_coefs.png")

alphas = np.logspace(-4, 4, 30)
l1_ratios = [0.1, 0.5, 0.9]
plt.figure(figsize=(7, 5))
for l1 in l1_ratios:
    coefs = []
    for a in alphas:
        enet = ElasticNet(alpha=a, l1_ratio=l1, random_state=42, max_iter=5000)
        pipe = Pipeline(steps=[("preprocessor", preprocessor), ("model", enet)])
        pipe.fit(X_train, y_train)
        coefs.append(pipe.named_steps["model"].coef_.ravel())
    plt.plot(alphas, np.mean(np.abs(coefs), axis=1), label=f"l1_ratio={l1}")

plt.xscale('log')
plt.title("مسیر ضرایب ElasticNet (میانگین قدر مطلق)")
plt.xlabel("alpha")
plt.ylabel("میانگین قدر مطلق ضرایب")
plt.legend()
plt.tight_layout()
plt.savefig("outputs/figures/elasticnet_path.png")
plt.show()
print("توضیح: ترکیب l1 و l2 رفتار ضرایب را کنترل می‌کند.")


In [ ]:

# ذخیره مدل
os.makedirs("outputs/models", exist_ok=True)
joblib.dump(best_model, "outputs/models/elasticnet.pkl")
print("مدل ElasticNet ذخیره شد.")


### 6-5) SGDRegressor


In [ ]:
model = SGDRegressor(random_state=42)
pipe = Pipeline(steps=[("preprocessor", preprocessor), ("model", model)])

param_grid = {
    "model__penalty": ["l2", "l1", "elasticnet"],
    "model__alpha": np.logspace(-6, -1, 10),
    "model__l1_ratio": [0.15, 0.3, 0.5, 0.7, 0.9],
    "model__max_iter": [2000, 5000],
    "model__learning_rate": ["optimal", "invscaling", "adaptive"],
}
cv = KFold(n_splits=5, shuffle=True, random_state=42)

random = RandomizedSearchCV(
    pipe,
    param_distributions=param_grid,
    n_iter=20,
    scoring="neg_root_mean_squared_error",
    cv=cv,
    random_state=42,
)
random.fit(X_train, y_train)

best_model = random.best_estimator_
print("بهترین پارامترها:", random.best_params_)

train_pred = best_model.predict(X_train)
test_pred = best_model.predict(X_test)

train_mae, train_rmse, train_r2 = regression_metrics(y_train, train_pred)
test_mae, test_rmse, test_r2 = regression_metrics(y_test, test_pred)

cv_rmse = -random.best_score_
cv_r2 = cross_val_score(best_model, X_train, y_train, scoring="r2", cv=cv)

results_df = pd.DataFrame({
    "Metric": ["MAE", "RMSE", "R2"],
    "Train": [train_mae, train_rmse, train_r2],
    "Test": [test_mae, test_rmse, test_r2],
})

cv_summary = pd.DataFrame({
    "Metric": ["RMSE_CV", "R2_CV"],
    "Mean": [cv_rmse, cv_r2.mean()],
    "Std": [0.0, cv_r2.std()],
})

save_table(results_df, "outputs/tables/sgd_results.csv")
save_table(cv_summary, "outputs/tables/sgd_cv_summary.csv")

save_results("SGDRegressor", {
    "train_mae": train_mae,
    "train_rmse": train_rmse,
    "train_r2": train_r2,
    "test_mae": test_mae,
    "test_rmse": test_rmse,
    "test_r2": test_r2,
    "cv_rmse_mean": cv_rmse,
    "cv_rmse_std": 0.0,
    "cv_r2_mean": cv_r2.mean(),
    "cv_r2_std": cv_r2.std(),
    "best_params": random.best_params_,
})

residuals = y_test - test_pred
plot_predicted_vs_actual(y_test, test_pred, "واقعی در برابر پیش‌بینی (SGD)", "outputs/figures/sgd_pred_vs_actual.png")
plot_residuals(y_test, test_pred, "Residuals (SGD)", "outputs/figures/sgd_residuals.png")
plot_residual_hist(residuals, "Histogram Residuals (SGD)", "outputs/figures/sgd_residual_hist.png")

feature_names = best_model.named_steps["preprocessor"].get_feature_names_out()
plot_top_coefficients(best_model.named_steps["model"], feature_names, "Top 15 ضرایب (SGD)", "outputs/figures/sgd_top_coefs.png")


In [ ]:

# ذخیره مدل
os.makedirs("outputs/models", exist_ok=True)
joblib.dump(best_model, "outputs/models/sgd_regressor.pkl")
print("مدل SGDRegressor ذخیره شد.")


## 7) مقایسه نهایی مدل‌ها


In [ ]:
with open("outputs/tables/results.json", "r", encoding="utf-8") as f:
    results = json.load(f)

results_df = pd.DataFrame(results).T
results_df = results_df.reset_index().rename(columns={"index": "Model"})
results_df = results_df.sort_values(by="test_rmse")

save_table(results_df, "outputs/tables/model_comparison.csv")

plt.figure(figsize=(8, 5))
plt.bar(results_df["Model"], results_df["test_rmse"])
plt.title("مقایسه RMSE مدل‌ها")
plt.ylabel("RMSE")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("outputs/figures/comparison_rmse.png")
plt.show()
print("توضیح: مدل با RMSE کمتر عملکرد بهتری دارد.")

plt.figure(figsize=(8, 5))
plt.bar(results_df["Model"], results_df["test_mae"])
plt.title("مقایسه MAE مدل‌ها")
plt.ylabel("MAE")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("outputs/figures/comparison_mae.png")
plt.show()
print("توضیح: MAE پایین‌تر نشان‌دهنده خطای کمتر است.")


## 8) جمع‌بندی نهایی
- **بهترین مدل برای دقت:** مدلی با کمترین RMSE تست.
- **بهترین مدل برای تفسیرپذیری:** معمولاً LinearRegression یا Ridge.
- **بهترین مدل برای پایداری:** مدلی با کمترین انحراف معیار در CV.

**پیشنهادهای بهبود:**
- مهندسی ویژگی (Feature Engineering)
- استفاده از RobustScaler
- مدیریت و حذف Outlierها
- بررسی مدل‌های غیرخطی (صرفاً به‌عنوان پیشنهاد)
